# Daten Vorbereitung 
### Ziele der Datenvorbereitung (Data Prep)
* **Data Analysis:** Fokus auf **menschliche Interpretierbarkeit** (Saubere Labels, korrekte Daten, klare Charts).
* **Data Science:** Fokus auf **Maschinen-Bereitschaft** (Skalierung, Encoding, Feature Engineering).

In [ ]:
import pandas as pd

# 
df = pd.read_csv('data.csv')

# First step: Look at the chaos
df.head(10)

### 🔍 Daten-Check: Erst schauen, dann coden!

Bevor wir zum nächsten Schritt übergehen, schau dir die Daten genau an. Du kannst die CSV-Datei direkt in **Excel** oder in **VS Code** (mit der *CSV Viewer* Extension) öffnen.

**Deine Aufgabe:**
1. Notiere dir, welche **Probleme** (Fehler, Lücken, Unstimmigkeiten) du in den Rohdaten beobachtest.
2. Überlege kurz, was wir tun könnten, um diese zu beheben (**Lösungsansätze**).

Komm anschließend hierher zurück. Wir werden die Probleme dann mit **Python-Funktionen** technisch verifizieren und die passenden Lösungen Schritt für Schritt umsetzen.

### 🔍 Initiales Daten-Audit
1. **Head:** `df.head()` für einen schnellen visuellen Scan.
2. **Datentypen:** `df.info()`, um falsche Formate zu finden.
3. **Outlier:** `df.describe()`, um unmögliche Min/Max-Werte zu entdecken.
4. **Lücken:** `df.isna().sum()`, um Datenlücken zu lokalisieren.
5. **Duplikate:** `df.duplicated().sum()`, um redundante Zeilen zu finden.
6. **Kategorien:** `.value_counts()`, um inkonsistente Labels (z. B. Groß-/Kleinschreibung) zu finden.

In [ ]:
#head
df.head(2)

### Erste Eindrücke

- Insgesamt haben wir 6 Spalten (Timestamp, Symbol, Open, High, Low, Close, Volume).
- Ich muss beim Timestamp auf das Format achten (Zeitstempel).
- Bei Preis und Volumen könnten Ausreißer ein Problem sein.
- Nullwerte vorhanden.

Das Volume zeigt mir die Trades an einem Tag, aber ich weiß nicht, wie viel davon bei hohen Preisen stattfanden. Ein kleiner Trade z. B. bei hohem Kurs kann meine Sicht auf den Preis stark beeinflussen, da er den Höchstwert nach oben treibt.

In [ ]:
#Info
df.info()

- In der Spalte `Close` gibt es drei Nullwerte; `Open` ist als Object klassifiziert, sollte aber ein Float sein. Das deutet auf einen Fehler hin – vermutlich steht dort ein Text oder eine andere unpassende Zeichenkette statt einer Zahl.

In [ ]:
# convert to numeric, coercing invalid values to NaN / # The 'coerce' parameter converts invalid values to NaN instead of raising an error

num = pd.to_numeric(df['Open'], errors='coerce')

# mask for the bad rows
mask = num.isna()

# show the offending entries (and/or the whole row)
df.loc[mask, 'Open']
# or: df.loc[mask]    # to see the entire row


In [ ]:
# show the entry at index 9
df.loc[9]

In [ ]:
import numpy as np 
# change the Open price in row 9
df.loc[9, 'Open'] = np.nan

# or change several columns in that row at once
df.loc[9, ['Timestamp','Symbol','Open','High','Low','Close','Volume']] = [
    '2026‑01‑10', 'AMZN', np.nan, 165.9, 203.0, 146.7, 160.0
]

# verify
df.loc[9]

In [ ]:
import re
from datetime import datetime

def guess_type_and_format(s):
    s = str(s).strip()

    # --- DATE FORMATS ---
    date_formats = [
        ("%Y-%m-%d", r"^\d{4}-\d{2}-\d{2}$"),      # 1999-09-09
        ("%d/%m/%Y", r"^\d{2}/\d{2}/\d{4}$"),      # 09/12/2023
        ("%m/%d/%Y", r"^\d{2}/\d{2}/\d{4}$"),      # US style
        ("%Y.%m.%d", r"^\d{4}\.\d{2}\.\d{2}$"),    # 2024.02.26
        ("%d.%m.%Y", r"^\d{2}\.\d{2}\.\d{4}$"),    # 22.02.2020  ← ADDED
    ]

    for fmt, pattern in date_formats:
        if re.fullmatch(pattern, s):
            try:
                dt = datetime.strptime(s, fmt)
                return {"type": "date", "standard": dt.strftime("%Y-%m-%d")}
            except:
                continue

    # --- TIME FORMATS ---
    time_formats = [
        ("%H:%M", r"^\d{2}:\d{2}$"),
        ("%H:%M:%S", r"^\d{2}:\d{2}:\d{2}$"),
    ]

    for fmt, pattern in time_formats:
        if re.fullmatch(pattern, s):
            try:
                dt = datetime.strptime(s, fmt)
                return {"type": "time", "standard": dt.strftime("%H:%M:%S")}
            except:
                continue

    # --- NUMERIC ---
    if s.isdigit():
        return {"type": "integer", "standard": int(s)}

    try:
        return {"type": "float", "standard": float(s)}
    except:
        pass

    return {"type": "string", "standard": s}

results = df["Timestamp"].astype(str).apply(guess_type_and_format)

df["type"] = results.apply(lambda x: x["type"])
df["Timestamp_standard"] = results.apply(lambda x: x["standard"])

In [ ]:
df.head(10)

In [ ]:
# Convert Timestamp_standard column to datetime object
df['Timestamp_standard'] = pd.to_datetime(df['Timestamp_standard'], errors='coerce')

In [ ]:
df.head()

In [ ]:
df.info()

# Fehlende Werte

There are **two main ways** to handle missing values in a column:
1. **Drop the missing rows**
2. **Fill (impute) the missing values**

In [ ]:
# checking data index 8 to 10 open column index 9 is NaN
df.loc[8:10]

In [ ]:

# After following there will be no index 9 as it is NaN dropna will remove it
df['Open'] = pd.to_numeric(df['Open'], errors='coerce')
df_dropna = df.dropna(subset=['Open'])

df_dropna.loc[8:10]

In [ ]:
#compare with df main dataframe
df.loc[8:10]

Jetzt die zweite Lösung, mit einem Imputer

In [ ]:
# Imputing Missing Values with Column Average

def fill_nan_with_average(df, column):
    """
    Replace NaN values in a column with the average of non-NaN values.
    Uses only basic math operations, no additional libraries.
    """
    # Calculate average manually
    col_data = df[column]
    non_nan_sum = 0
    non_nan_count = 0
    
    for value in col_data:
        if pd.notna(value):
            non_nan_sum += value
            non_nan_count += 1
    
    average = non_nan_sum / non_nan_count if non_nan_count > 0 else 0
    
    # Replace NaN with average
    for idx in df.index:
        if pd.isna(df.at[idx, column]):
            df.at[idx, column] = average
    
    return df

# Usage
df_filled = fill_nan_with_average(df.copy(), 'Open')
df_filled.loc[8:10]


Machen wir das Gleiche für die Spalte `Close`

In [ ]:
df.head()

In [ ]:
df_filled = fill_nan_with_average(df.copy(), 'Close')
df_filled.head()

Du kannst df_dropna (gelöschte Zeilen) und df_imputed (imputierte Zeilen) mit dem ursprünglichen df vergleichen, um zu entscheiden, welche Strategie für deine Analyse am besten geeignet ist.

# Ausreißer / Outliers  

In [ ]:
#Outlier
df.describe()

In [ ]:
# Find where Close column has the value 99999.00
outlier_close = df[df['Close'] == 99999.00]
print(outlier_close)

# Or get just the index
outlier_idx = df[df['Close'] == 99999.00].index.tolist()
print(f"Index of outlier: {outlier_idx}")

### Ausreißererkennung mit Z-Score (Aktien: Open / Close)

Die Z-Score-Methode erkennt Werte, die weit vom Mittelwert entfernt sind.

Formel:
Z = (Wert − Mittelwert) / Standardabweichung

Regel:

Als Ausreißer markieren, wenn |Z| > 3

Eignet sich das für Open-/Close-Daten von Aktien?

✔ Funktioniert, wenn die Renditen ungefähr normalverteilt sind

✘ Rohaktienpreise sind oft verzerrt und zeigen Trends über die Zeit

⚠ Besser, den Z-Score auf Renditen, nicht auf Rohpreise, anzuwenden

Für einfache Datenbereinigung ist es akzeptabel.
Für finanzielle Analysen sollte man den Z-Score auf prozentuale Renditen anwenden.

In [ ]:
import numpy as np
import pandas as pd

cols = ['Open', 'Close', 'High', 'Low', 'Volume']

# Ensure numeric
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# Compute Z-scores (temporary, not stored in df)
z_scores = (df[cols] - df[cols].mean()) / df[cols].std()

# Boolean mask of outliers
outlier_mask = z_scores.abs() > 3

# Keep only rows with at least one outlier
df_outliers = df[outlier_mask.any(axis=1)].copy()

# Add information: which column(s) had the outlier
df_outliers["Outlier_Columns"] = outlier_mask[outlier_mask.any(axis=1)] \
    .apply(lambda row: list(row[row].index), axis=1)

df_outliers.head()

# Umgang mit Outliers/Ausreißern

Wir haben Ausreißer in `Open`, `Close`, `High`, `Low`, `Volume` erkannt.  

Es gibt **zwei einfache Möglichkeiten**, damit umzugehen:

1. **Methode 1: Ausreißer z. B. mit dem Median ersetzen**  
2. **Methode 2: Zeilen mit Ausreißern löschen**

In [ ]:
#Aktionsplan Outlier: Unmögliche Werte über Boolean Indexing filtern (z.B. `df[df['Preis'] < limit]`).


cols = ['Open', 'Close', 'High', 'Low', 'Volume']

# Ensure numeric
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# Compute Z-scores (detect outliers)
z_scores = (df[cols] - df[cols].mean()) / df[cols].std()
outlier_mask = z_scores.abs() > 3  # True if value is an outlier

# Replace outliers with column mean
df_mean_imputed = df.copy()
for col in cols:
    mean_val = df_mean_imputed[col].mean()  # compute mean of the column
    df_mean_imputed[col] = df_mean_imputed[col].mask(outlier_mask[col], mean_val)

df_mean_imputed.loc[18:20]

In [ ]:
# Keep only rows without any outlier
df_dropped = df[~outlier_mask.any(axis=1)].copy()

df_dropped.loc[18:20]

In [ ]:
# Duplicates: look for duplicate rows (all columns)
dups = df[df.duplicated(keep=False)]
print(f"duplicates found: {len(dups)}")
dups



In [ ]:

# Aktionsplan Duplicates: `drop_duplicates()` für saubere, einzigartige Datensätze# remove them – either create a new frame or do it in‑place
df_no_dups = df.drop_duplicates()         # keeps first occurrence
# or: df.drop_duplicates(inplace=True)

df_no_dups.head()

## Kategorien

In [ ]:
# Categories
df['Symbol'].value_counts()



In [ ]:
# Example: convert 'Symbol' column to uppercase
df['Symbol'] = df['Symbol'].str.upper()

In [ ]:
df['Symbol'].value_counts()
